In [12]:
import google.generativeai as genai
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
import re

# 1. Configuration for Gemini
# IMPORTANT: Paste your copied API key inside the quotes below!
GEMINI_API_KEY = "YOUR_API_KEY_HERE" 

try:
    genai.configure(api_key="AIzaSyBkziReCGWy4_*************")
    # Using the flash-preview model for high-speed coding tasks
    model = genai.GenerativeModel('gemini-3-flash-preview')
except Exception as e:
    print(f"Configuration Error: {e}")

# 2. Function to talk to Gemini
def ask_gemini(prompt, role_instruction):
    full_prompt = f"System Role: {role_instruction}\n\nTask:\n{prompt}"
    try:
        response = model.generate_content(full_prompt)
        return response.text
    except Exception as e:
        return f"**Error connecting to Gemini API:** {str(e)}. Check your API key!"

# 3. Function to extract PlantUML and save it
def save_plantuml(text, filename="output_diagram.plantuml"):
    match = re.search(r'(@startuml.*?@enduml)', text, re.DOTALL)
    if match:
        puml_code = match.group(1)
        with open(filename, "w") as file:
            file.write(puml_code)
        return True, f"✅ PlantUML code successfully extracted and saved to **{filename}**"
    else:
        return False, "⚠️ Could not find @startuml and @enduml tags in the output."

# 4. Building the UI Elements
title = widgets.HTML("<h2>Enterprise AI Architecture Pipeline (Powered by Gemini)</h2>")
instruction = widgets.HTML("<p>Enter your project description below:</p>")
text_input = widgets.Textarea(
    value='',
    placeholder='Type your requirements here...',
    layout=widgets.Layout(width='100%', height='100px')
)
run_button = widgets.Button(
    description='Generate Pipeline',
    button_style='success'
)
output_area = widgets.Output()

# 5. The Main Pipeline Logic
def on_button_click(b):
    with output_area:
        clear_output() 
        user_input = text_input.value
        
        if not user_input.strip():
            display(Markdown("**Please enter a description first!**"))
            return

        display(Markdown("### 🚀 Starting Gemini Pipeline... \n---"))

        # STAGE 1: Requirements
        display(Markdown("#### ⏳ Stage 1: Generating Requirements..."))
        stage1_prompt = f"Based on this project: '{user_input}', generate Elicitation Techniques, Functional Requirements, and Non-Functional Requirements."
        stage1_output = ask_gemini(stage1_prompt, "You are an expert IT Business Analyst.")
        display(Markdown(stage1_output))
        display(Markdown("\n---\n"))

        # STAGE 2: User Stories
        display(Markdown("#### ⏳ Stage 2: Generating User Stories..."))
        stage2_prompt = f"Based strictly on the following requirements, generate Agile User Stories.\n\nRequirements:\n{stage1_output}"
        stage2_output = ask_gemini(stage2_prompt, "You are an expert Agile Product Owner.")
        display(Markdown(stage2_output))
        display(Markdown("\n---\n"))

        # STAGE 3: PlantUML Activity Diagram
        display(Markdown("#### ⏳ Stage 3: Generating PlantUML Activity Diagram..."))
        stage3_prompt = f"""Based on the Requirements and User Stories, generate PlantUML code for an ACTIVITY Diagram showing the workflow of a user requesting and using the GPU Data Center.

        RULES:
        1. MUST enclose code within @startuml and @enduml tags.
        2. Use ONLY proper Activity Diagram syntax. Every action MUST start with a colon and end with a semicolon (e.g., :Provision GPU Node;).
        3. DO NOT use Sequence Diagram syntax (like 'note right of' or 'participant').
        4. Use swimlanes to separate different actors/systems.
        5. Follow this exact syntax structure:

           @startuml
           title Data Center Allocation Workflow
           
           |AI Engineer|
           start
           :Request GPU Resources;
           
           |System|
           if (Resources Available?) then (Yes)
             :Allocate vRAM;
           else (No)
             :Add to Queue;
             stop
           endif
           
           fork
             :Deploy LLM Model;
           fork again
             :Mount NVMe Storage;
           end fork
           stop
           @enduml

        Map the workflow of the Data Center, incorporating the User Stories generated in the previous step.
        
        User Stories/Requirements for mapping:
        {stage2_output}"""
        
        stage3_output = ask_gemini(stage3_prompt, "You are an expert Cloud Architect. Only output valid PlantUML Activity Diagram code.")
        display(Markdown(stage3_output))
        display(Markdown("\n---\n"))

        # STAGE 4: Save to File
        display(Markdown("#### 💾 Saving File..."))
        success, message = save_plantuml(stage3_output)
        display(Markdown(message))

run_button.on_click(on_button_click)
display(title, instruction, text_input, run_button, output_area)

HTML(value='<h2>Enterprise AI Architecture Pipeline (Powered by Gemini)</h2>')

HTML(value='<p>Enter your project description below:</p>')

Textarea(value='', layout=Layout(height='100px', width='100%'), placeholder='Type your requirements here...')

Button(button_style='success', description='Generate Pipeline', style=ButtonStyle())

Output()